# 07 — XGBoost global vs LightGBM global y modelos finales

La propuesta prometía XGBoost; en el panel del 05 el boosting fue LightGBM. Este notebook corre **XGBoost global** en el mismo panel (39 municipios, mismas variables, mismos pliegues 2016–2025, mismo presupuesto de ajuste: 3 configuraciones elegidas por PR-AUC en validación y umbral que maximiza F1) y lo compara con LightGBM con las mismas pruebas pareadas del 05.

También deja en código el **reentrenamiento final** de los dos modelos con todos los datos (hiperparámetro = moda de los pliegues; árboles y umbral = mediana).

**Regla de decisión (fijada antes de correr):** se elige XGBoost como modelo final si no es significativamente peor que LightGBM en F1 ni en PR-AUC (IC 95 % bootstrap de la diferencia que incluye 0 o la favorece), porque es el modelo de la propuesta. Si es peor, se mantiene LightGBM.

Reutiliza las secciones 1–9 del notebook 05 (no las copia, las ejecuta), así que necesita el 05 y los datos en Drive. No escribe en las carpetas del 05.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, zipfile, shutil, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat', 'lightgbm', 'xgboost', 'python-calamine'], check=True)
import nbformat, numpy as np, pandas as pd, xgboost as xgb, lightgbm as lgb
from scipy import stats
from sklearn.metrics import (f1_score, average_precision_score, roc_auc_score, recall_score,
                             precision_score, matthews_corrcoef, brier_score_loss)

NB05 = os.environ.get('DENGUE_NB05', '/content/drive/MyDrive/Colab Notebooks/05_dl_profundo_dengue.ipynb')
SAL = os.environ.get('DENGUE_SALIDA_07', '/content/drive/MyDrive/Datos/dengue/comparacion_xgboost')
os.environ['DENGUE_SALIDA'] = '/content/salida_05_tmp'      # las celdas del 05 escriben aquí, no en sus carpetas
ZIP = '/content/drive/MyDrive/export_xgb.zip'                # copia opcional de resultados (si el archivo existe)

nb = nbformat.read(NB05, as_version=4)
C = [c.source for c in nb.cells if c.cell_type == 'code']
fin = next(i for i, s in enumerate(C) if 'def lightgbm_global' in s)
G = {'get_ipython': get_ipython, '__name__': '__main__'}
for i, s in enumerate(C[:fin + 1]):                          # secciones 1–9 del 05
    exec(compile(s, f'05_celda_{i}', 'exec'), G)
import matplotlib.pyplot as plt; plt.close('all')
assert G['FRECUENCIA'] == 'mensual' and len(G['MUNIS']) == 39
tabla, X, SEED, MUNIS = G['tabla'], G['X_LGB'], G['SEED'], G['MUNIS']
os.makedirs(SAL, exist_ok=True)
print('xgboost', xgb.__version__, '| lightgbm', lgb.__version__, '|', len(tabla), 'filas |', len(X), 'variables')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Keras backend: tensorflow | RAPIDO: False
GPU: no hay (Entorno de ejecución → Cambiar tipo → GPU T4 para ir más rápido)
Leído /content/drive/MyDrive/Datos/dengue/procesado/casos_limpios.parquet
1,700,889 casos | 1,115 municipios | 2007-01-01 a 2025-12-31
Frecuencia: mensual | salida: /content/salida_05_tmp/mensual
228 periodos × 1115 municipios
39 municipios con canal estable (<20% de meses con <5 casos)
ONI descargado: 1950-02 a 2026-10
Proporción de periodos en exceso (2012+): mediana 0.33 | rango 0.16 - 0.62


6,045 filas | 39 municipios | 2012–2025 | prevalencia de exceso 0.34 | ONI: sí
xgboost 3.4.1 | lightgbm 4.6.0 | 6045 filas | 20 variables


## XGBoost global
Misma complejidad que la grilla de LightGBM (7/15/31 hojas ≈ profundidad 3/4/5), mismo peso de la clase positiva (`scale_pos_weight` ≈ `is_unbalance`), parada temprana con 50 rondas y municipio como variable categórica.

In [2]:
XGB_GRID = [dict(max_depth=3, min_child_weight=10), dict(max_depth=4, min_child_weight=5),
            dict(max_depth=5, min_child_weight=2)]
XGB_BASE = dict(objective='binary:logistic', eval_metric='logloss', eta=0.03, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=5.0, tree_method='hist', max_cat_to_onehot=1, seed=SEED)
CATS = list(range(len(MUNIS)))

def dmat(d):
    Xd = d[X].copy()
    Xd['mun_id'] = pd.Categorical(Xd['mun_id'], categories=CATS)
    return xgb.DMatrix(Xd, label=d['y'].values, enable_categorical=True)

def peso(d):
    return float((d['y'] == 0).sum() / max(int(d['y'].sum()), 1))

def xgboost_global(tr, te, itr, iva):
    dtr, dva = dmat(itr), dmat(iva)
    mejor = None
    for g in XGB_GRID:
        b = xgb.train({**XGB_BASE, **g, 'scale_pos_weight': peso(itr)}, dtr, num_boost_round=1500,
                      evals=[(dva, 'va')], early_stopping_rounds=50, verbose_eval=False)
        n = b.best_iteration + 1
        p = b.predict(dva, iteration_range=(0, n))
        ap = average_precision_score(iva['y'], p)
        if mejor is None or ap > mejor[0]:
            mejor = (ap, g, max(n, 20), p)
    _, g, n_arb, p_val = mejor
    u = G['elegir_umbral'](iva['y'], p_val)
    b = xgb.train({**XGB_BASE, **g, 'scale_pos_weight': peso(tr)}, dmat(tr), num_boost_round=n_arb)
    return b.predict(dmat(te)), u, {'max_depth': g['max_depth'], 'arboles': n_arb}

## Validación con ventana creciente (igual que la sección 11 del 05)

In [3]:
MODS = {'Persistencia': G['persistencia'], 'LightGBM global': G['lightgbm_global'], 'XGBoost global': xgboost_global}
preds, info = [], []
for Y in G['ANIOS_TEST']:
    tr, te, itr, iva = G['particion'](Y)
    if len(te) == 0 or iva['y'].sum() == 0:
        print(Y, 'sin datos suficientes, se omite'); continue
    base = te[['mun', 'anio_o', 'fecha_t', 'y', 'exc_o']].copy()
    seg = {}
    for n, fn in MODS.items():
        t0 = time.time(); p, u, extra = fn(tr, te, itr, iva); seg[n] = round(time.time() - t0, 1)
        base[f'p|{n}'] = p; base[f'u|{n}'] = u
        info.append({'anio': Y, 'modelo': n, 'umbral': u, 'segundos': seg[n], **extra})
    preds.append(base)
    print(Y, f'test {len(te):,} ({te.y.mean():.0%} exceso)', seg)
PR = pd.concat(preds, ignore_index=True)
H = pd.DataFrame(info)

# Se agregan las predicciones del 05 (binomial negativa y redes) para comparar en los mismos meses
O = pd.read_csv(f"{G['BASE']}/dengue/comparacion_dl_profundo/mensual/predicciones_test.csv", dtype={'mun': str})
for d in (PR, O):
    d['mun'] = d['mun'].astype(str).str.zfill(5); d['fecha_t'] = pd.to_datetime(d['fecha_t'])
otros = [c for c in O.columns if c.startswith(('p|', 'u|')) and not any(k in c for k in ('LightGBM', 'Persistencia'))]
PR = PR.merge(O[['mun', 'fecha_t', 'y', 'p|LightGBM global'] + otros]
              .rename(columns={'y': 'y_05', 'p|LightGBM global': 'p_lgb_05'}), on=['mun', 'fecha_t'], how='left')
assert PR['y_05'].notna().all() and (PR['y'] == PR['y_05']).all(), 'las filas no coinciden con el 05'
print('LightGBM reproduce al 05: correlación', round(np.corrcoef(PR['p|LightGBM global'], PR['p_lgb_05'])[0, 1], 4),
      '| dif. máx.', round((PR['p|LightGBM global'] - PR['p_lgb_05']).abs().max(), 4))
PR = PR.drop(columns=['y_05', 'p_lgb_05'])
PR.to_csv(f'{SAL}/predicciones_test_07.csv', index=False); H.to_csv(f'{SAL}/hiperparametros_por_pliegue_07.csv', index=False)

2016 test 457 (14% exceso) {'Persistencia': 0.0, 'LightGBM global': 0.5, 'XGBoost global': 0.7}
2017 test 468 (4% exceso) {'Persistencia': 0.0, 'LightGBM global': 1.0, 'XGBoost global': 1.3}
2018 test 468 (20% exceso) {'Persistencia': 0.0, 'LightGBM global': 2.2, 'XGBoost global': 7.4}
2019 test 468 (54% exceso) {'Persistencia': 0.0, 'LightGBM global': 0.7, 'XGBoost global': 1.1}
2020 test 468 (13% exceso) {'Persistencia': 0.0, 'LightGBM global': 2.1, 'XGBoost global': 9.9}
2021 test 466 (16% exceso) {'Persistencia': 0.0, 'LightGBM global': 18.7, 'XGBoost global': 13.0}
2022 test 456 (25% exceso) {'Persistencia': 0.0, 'LightGBM global': 2.5, 'XGBoost global': 4.0}
2023 test 456 (50% exceso) {'Persistencia': 0.0, 'LightGBM global': 1.1, 'XGBoost global': 5.4}
2024 test 450 (69% exceso) {'Persistencia': 0.0, 'LightGBM global': 2.0, 'XGBoost global': 2.9}
2025 test 350 (28% exceso) {'Persistencia': 0.0, 'LightGBM global': 1.9, 'XGBoost global': 2.6}
LightGBM reproduce al 05: correlación 1

## Métricas y pruebas pareadas (mismo método que la sección 13 del 05)

In [4]:
NOMBRES = ['Persistencia', 'Binomial negativa local', 'LightGBM global', 'XGBoost global',
           'GRU global (1 capa)', 'LSTM global (2 capas)', 'TCN global']
y = PR['y'].values; ini = (y == 1) & (PR['exc_o'].values == 0)

def metricas(n):
    p = PR[f'p|{n}'].values; a = (p >= PR[f'u|{n}'].values).astype(int)
    b = np.minimum((p * 10).astype(int), 9)
    ece = sum(abs(p[b == k].mean() - y[b == k].mean()) * (b == k).sum() for k in range(10) if (b == k).any()) / len(y)
    return {'F1': f1_score(y, a), 'PR-AUC': average_precision_score(y, p), 'ROC-AUC': roc_auc_score(y, p),
            'Sensibilidad': recall_score(y, a), 'Precision': precision_score(y, a),
            'Especificidad': ((a == 0) & (y == 0)).sum() / (y == 0).sum(), 'MCC': matthews_corrcoef(y, a),
            'Inicios anticipados': a[ini].mean(), 'Falsas alarmas': a[y == 0].mean(),
            'Brier': brier_score_loss(y, p), 'ECE': ece}

MET = pd.DataFrame({n: metricas(n) for n in NOMBRES}).T
MET.round(3).to_csv(f'{SAL}/metricas_07.csv')
anual = pd.concat({Yr: pd.DataFrame({n: {'F1': f1_score(g.y, g[f'p|{n}'] >= g[f'u|{n}']),
                                         'PR-AUC': average_precision_score(g.y, g[f'p|{n}'])}
                                     for n in ['Persistencia', 'LightGBM global', 'XGBoost global']}).T
                   for Yr, g in PR.groupby('anio_o')}, names=['anio', 'modelo'])
anual.round(3).to_csv(f'{SAL}/metricas_por_anio_07.csv')

def por_bloque(n):
    out = []
    for (m, Yr), g in PR.groupby(['mun', 'anio_o']):
        p = g[f'p|{n}'].values; yy = g['y'].values
        out.append({'mun': m, 'anio': Yr, 'brier': np.mean((p - yy) ** 2),
                    'pr_auc': average_precision_score(yy, p) if 0 < yy.sum() < len(yy) else np.nan})
    return pd.DataFrame(out).set_index(['mun', 'anio'])

BLOQ = {n: por_bloque(n) for n in NOMBRES}
bloques = list(PR.groupby(['mun', 'anio_o']).indices.values())
rng = np.random.default_rng(SEED)
muestras = [np.concatenate([bloques[i] for i in rng.integers(0, len(bloques), len(bloques))]) for _ in range(2000)]

def comparar(a, b):
    da, db = BLOQ[a], BLOQ[b]
    w_br = stats.wilcoxon(da['brier'], db['brier'])
    ok = da['pr_auc'].notna() & db['pr_auc'].notna()
    w_ap = stats.wilcoxon(da.loc[ok, 'pr_auc'], db.loc[ok, 'pr_auc'])
    pa, pb = PR[f'p|{a}'].values, PR[f'p|{b}'].values
    ea, eb = (pa >= PR[f'u|{a}'].values).astype(int), (pb >= PR[f'u|{b}'].values).astype(int)
    d_f1 = [f1_score(y[i], ea[i]) - f1_score(y[i], eb[i]) for i in muestras]
    d_ap = [average_precision_score(y[i], pa[i]) - average_precision_score(y[i], pb[i]) for i in muestras]
    return {'A': a, 'B': b, 'dif_F1': f1_score(y, ea) - f1_score(y, eb),
            'F1_IC_inf': np.percentile(d_f1, 2.5), 'F1_IC_sup': np.percentile(d_f1, 97.5),
            'dif_PR-AUC': average_precision_score(y, pa) - average_precision_score(y, pb),
            'PRAUC_IC_inf': np.percentile(d_ap, 2.5), 'PRAUC_IC_sup': np.percentile(d_ap, 97.5),
            'Brier_mejor_en_A': (da['brier'] < db['brier']).mean(), 'p_Wilcoxon_Brier': w_br.pvalue,
            'PRAUC_mejor_en_A': (da.loc[ok, 'pr_auc'] > db.loc[ok, 'pr_auc']).mean(), 'p_Wilcoxon_PRAUC': w_ap.pvalue}

def holm(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    adj = np.minimum(np.maximum.accumulate((m - np.arange(m)) * p[o]), 1.0)
    out = np.empty(m); out[o] = adj
    return out

pares = [('XGBoost global', 'LightGBM global'), ('XGBoost global', 'GRU global (1 capa)'),
         ('XGBoost global', 'Binomial negativa local'), ('XGBoost global', 'Persistencia'),
         ('LightGBM global', 'Persistencia')]
PRU = pd.DataFrame([comparar(a, b) for a, b in pares])
for c in ['p_Wilcoxon_Brier', 'p_Wilcoxon_PRAUC']:
    PRU[c.replace('Wilcoxon', 'Holm')] = holm(PRU[c])
PRU.round(4).to_csv(f'{SAL}/pruebas_pareadas_07.csv', index=False)

r = PRU.iloc[0]
no_peor = (r['F1_IC_sup'] >= 0) and (r['PRAUC_IC_sup'] >= 0)
DECISION = 'XGBoost global' if no_peor else 'LightGBM global'
print(MET.round(3).to_string()); print(); print(PRU.round(3).T.to_string())
print('\nDECISIÓN (regla fijada antes de correr):', DECISION)

                            F1  PR-AUC  ROC-AUC  Sensibilidad  Precision  Especificidad    MCC  Inicios anticipados  Falsas alarmas  Brier    ECE
Persistencia             0.742   0.625    0.818         0.745      0.739          0.892  0.635                0.000           0.108  0.151  0.151
Binomial negativa local  0.739   0.844    0.919         0.820      0.673          0.836  0.623                0.522           0.164  0.100  0.021
LightGBM global          0.756   0.837    0.919         0.838      0.689          0.845  0.648                0.466           0.155  0.104  0.055
XGBoost global           0.747   0.829    0.912         0.833      0.677          0.836  0.634                0.457           0.164  0.109  0.061
GRU global (1 capa)      0.762   0.855    0.928         0.800      0.727          0.876  0.659                0.367           0.124  0.102  0.066
LSTM global (2 capas)    0.748   0.824    0.910         0.845      0.671          0.829  0.636                0.475         

## Modelos finales (todos los datos, 2012–2025)
Hiperparámetro de complejidad = moda de los pliegues; árboles y umbral = mediana de los pliegues. Es la misma regla con la que se entrenó el LightGBM final del repo (221 árboles, 7 hojas, umbral 0,425); aquí queda en código.

In [5]:
mun_id = {m: i for i, m in enumerate(MUNIS)}
def guardar(carpeta, meta, imp):
    os.makedirs(carpeta, exist_ok=True)
    json.dump(meta, open(f'{carpeta}/metadata.json', 'w'), indent=2, ensure_ascii=False)
    imp.sort_values(ascending=False).rename('ganancia').to_csv(f'{carpeta}/importancia_variables.csv')

comun = {'filas': len(tabla), 'periodo_objetivo': '2012-2025', 'prevalencia_exceso': round(float(tabla['y'].mean()), 3),
         'municipios': len(MUNIS), 'mun_id': mun_id, 'variables': X, 'horizonte_meses': 2, 'oni': True,
         'entrenado_con': 'notebooks/07_xgboost_vs_lightgbm.ipynb'}

h = H[H.modelo == 'LightGBM global']
nl = int(h['num_leaves'].mode().iloc[0]); n_l = int(round(h['arboles'].median())); u_l = float(h['umbral'].median())
prm_l = {**G['LGB_BASE'], **next(g for g in G['LGB_GRID'] if g['num_leaves'] == nl)}
b_l = lgb.train(prm_l, lgb.Dataset(tabla[X], tabla['y'], categorical_feature=['mun_id']), num_boost_round=n_l)
d_l = f'{SAL}/modelos/lightgbm_global'; os.makedirs(d_l, exist_ok=True); b_l.save_model(f'{d_l}/lightgbm_global_final.txt')
guardar(d_l, {'modelo': 'LightGBM global', **comun, 'num_leaves': nl, 'arboles': n_l, 'umbral_decision': u_l,
              'hiperparametros': prm_l}, pd.Series(b_l.feature_importance('gain'), index=X))

h = H[H.modelo == 'XGBoost global']
md = int(h['max_depth'].mode().iloc[0]); n_x = int(round(h['arboles'].median())); u_x = float(h['umbral'].median())
prm_x = {**XGB_BASE, **next(g for g in XGB_GRID if g['max_depth'] == md), 'scale_pos_weight': peso(tabla)}
b_x = xgb.train(prm_x, dmat(tabla), num_boost_round=n_x)
d_x = f'{SAL}/modelos/xgboost_global'; os.makedirs(d_x, exist_ok=True); b_x.save_model(f'{d_x}/xgboost_global_final.json')
gx = b_x.get_score(importance_type='total_gain')
guardar(d_x, {'modelo': 'XGBoost global', **comun, 'max_depth': md, 'arboles': n_x, 'umbral_decision': u_x,
              'hiperparametros': prm_x, 'version_xgboost': xgb.__version__},
        pd.Series({v: gx.get(v, 0.0) for v in X}))
print('LightGBM final:', nl, 'hojas,', n_l, 'árboles, umbral', u_l, '| XGBoost final: profundidad', md, ',', n_x, 'árboles, umbral', u_x)
json.dump({'decision': DECISION, 'lightgbm': {'num_leaves': nl, 'arboles': n_l, 'umbral': u_l},
           'xgboost': {'max_depth': md, 'arboles': n_x, 'umbral': u_x}}, open(f'{SAL}/decision_07.json', 'w'), indent=2)

if os.path.exists(ZIP):                                       # exportación opcional en un solo archivo
    with zipfile.ZipFile('/content/x07.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for raiz, _, archivos in os.walk(SAL):
            for f in archivos:
                z.write(os.path.join(raiz, f), os.path.relpath(os.path.join(raiz, f), SAL))
    shutil.copyfile('/content/x07.zip', ZIP); print('zip', os.path.getsize(ZIP))
print('listo; salidas en', SAL)

LightGBM final: 7 hojas, 221 árboles, umbral 0.42500000000000004 | XGBoost final: profundidad 3 , 281 árboles, umbral 0.45
zip 367756
listo; salidas en /content/drive/MyDrive/Datos/dengue/comparacion_xgboost
